Test the manipulation of attention tensors

In [141]:
import torch
import torch.nn as nn

In [142]:
DEVICE = torch.device('cpu')
DTYPE = torch.float32

In [143]:
batch_size = 1
seq_len = 1
dim_size = 2
heads = 1
head_dim = dim_size // heads
assert dim_size % head_dim == 0
tensor = torch.randn(batch_size, seq_len, dim_size, device=DEVICE, dtype=DTYPE, requires_grad=False)
tensor.shape

torch.Size([1, 1, 2])

In [144]:
projections = tuple(nn.Linear(dim_size, dim_size, False, DEVICE, DTYPE).eval() for _ in range(4))
q_proj, k_proj, v_proj, out_proj = projections
projections

(Linear(in_features=2, out_features=2, bias=False),
 Linear(in_features=2, out_features=2, bias=False),
 Linear(in_features=2, out_features=2, bias=False),
 Linear(in_features=2, out_features=2, bias=False))

In [145]:
query, key, value = tuple(
    m(tensor).view(batch_size, seq_len, heads, head_dim) 
    for m in projections[:-1]
)
query, query.shape

(tensor([[[[ 0.3523, -0.0900]]]], grad_fn=<ViewBackward0>),
 torch.Size([1, 1, 1, 2]))

In [146]:
energy = torch.einsum("bqhd, bkhd -> bhqk", [query, query])
energy.shape, energy

(torch.Size([1, 1, 1, 1]), tensor([[[[0.1322]]]], grad_fn=<ViewBackward0>))

In [148]:
scores = torch.matmul(query.transpose(1, 2), query.transpose(1, 2).transpose(-2, -1))
scores.shape, scores

(torch.Size([1, 1, 1, 1]),
 tensor([[[[0.1322]]]], grad_fn=<UnsafeViewBackward0>))